In [1]:
import pandas as pd
import sqlite3

# 1. LOAD: Read the raw CSV (keeping the Nulls/NaNs)
df = pd.read_csv(r"C:\cabarrus-gis-prep\sql\data\Recreation_Facilities.csv")

# 2. CONNECT: Open the tunnel to your database folder
db_path = r"C:\cabarrus-gis-prep\sql\databases\cabarrus_recreation.db"
conn = sqlite3.connect(db_path)

# 3. SAVE: Pour the data into the 'parks' table
df.to_sql('parks', conn, if_exists='replace', index=False)

# 4. INSPECT: Grab every single column (the '*')
query = "SELECT * FROM parks LIMIT 5"
all_data = pd.read_sql_query(query, conn)

#MAKING SURE OUTPUTS ARE EASY TO READ
# SHOW ALL COLUMNS
pd.set_option('display.max_columns', None)

# SHOW FULL COLUMN WIDTH
pd.set_option('display.max_colwidth', None)

# MAKE OUTPUT WIDER
pd.set_option('display.width', 2000)

# SHOW MORE ROWS IF NEEDED
pd.set_option('display.max_rows', 100)

# 5. RESULTS
print(f"✅ Success! Loaded {len(all_data.columns)} columns.")
print("\n--- ALL COLUMN NAMES ---")
print(all_data.columns.tolist()) 

print("\n--- DATA PREVIEW ---")
print(all_data.head())

print("looking at all the unique jurrisdictions")

all_jurisdictions = df['NAME'].unique()
print(all_jurisdictions)



✅ Success! Loaded 33 columns.

--- ALL COLUMN NAMES ---
['X', 'Y', 'OBJECTID', 'FACILITYID', 'NAME', 'SUBTYPEFIELD', 'FEATURECODE', 'FULLADDR', 'OPERDAYS', 'OPERHOURS', 'PARKAREA', 'PARKURL', 'NUMPARKING', 'RESTROOM', 'ADACOMPLY', 'CAMPING', 'SWIMMING', 'HIKING', 'FISHING', 'PICNIC', 'BOATING', 'HUNTING', 'ROADCYCLE', 'MTBCYCLE', 'PLAYGROUND', 'GOLF', 'SKI', 'SOCCER', 'BASEBALL', 'BASKETBALL', 'TENNIS', 'Shelter', 'Football']

--- DATA PREVIEW ---
              X              Y  OBJECTID FACILITYID                                             NAME  SUBTYPEFIELD     FEATURECODE                             FULLADDR   OPERDAYS                                    OPERHOURS  PARKAREA                                                                                         PARKURL  NUMPARKING RESTROOM ADACOMPLY CAMPING SWIMMING HIKING FISHING PICNIC BOATING HUNTING ROADCYCLE MTBCYCLE PLAYGROUND GOLF SKI SOCCER BASEBALL BASKETBALL TENNIS Shelter Football
0  1.515174e+06  644107.563114         1  

In [7]:
"""The Request: "Khoi, since we don't have construction data, just give me a map of Playgrounds located at 
Schools in Mt Pleasant and Concord. I also want to see the FEATURECODE so I know if it's a School or a Park."
"""


query= """
SELECT NAME, FULLADDR, X, Y, FEATURECODE
FROM parks
WHERE PLAYGROUND = 'Yes' 
AND (NAME LIKE 'Mt Pleasant%' OR NAME LIKE '%Concord%')
AND (NAME LIKE'%School%')

"""

playgrounds = pd.read_sql_query(query, conn)

print(playgrounds.head())

                            NAME                     FULLADDR             X  \
0  Mt Pleasant Elementary School  8555 North Rd, 704-920-3350  1.573466e+06   

               Y  FEATURECODE  
0  609882.466292  School Park  


In [ ]:
"""Hey Khoi, I'm heading into a meeting and need two numbers. First, what is the total number of facilities we have in
 our inventory? Second, what is the total acreage of all those parks combined? I need to make sure our insurance coverage 
 matches the land we actually own.

"""

query = """

SELECT 
    COUNT(*) AS total_facilities,
    SUM(PARKAREA) AS total_area
FROM parks
WHERE PARKAREA IS NOT NULL AND PARKAREA >0


"""

stats = pd.read_sql_query(query,conn)

print(stats.head())

   total_facilities  total_area
0                57       648.0


In [3]:
"""
From: Ashley (Department Lead)
Subject: Breakdown of Park Inventory by City

"Khoi, those totals were great. Now, I need to know which cities are the most 'park-heavy.' Can you give me a list 
showing each Jurisdiction, the Count of parks they have, and their Total Acreage? Please rank them from the most acreage 
to the least so I can see our biggest liabilities first."



""" 


query = """
SELECT
    CASE
        WHEN NAME LIKE '%Concord%' THEN 'concord'
        WHEN NAME LIKE '%Kannapolis%' THEN 'kannapolis'
    END AS Jurisdiction,
    COUNT(*) AS park_count,
    SUM(PARKAREA) AS total_ParkArea
FROM parks
WHERE PARKAREA IS NOT NULL AND PARKAREA > 0 
GROUP BY Jurisdiction
ORDER BY total_ParkArea DESC


"""

#EXPLANATION OF THE QUERY:
# So here i first wrote ------- [[[(select CASE

#        WHEN NAME LIKE '%Concord%' THEN 'concord'

#        WHEN NAME LIKE '%Kannapolis%' THEN 'kannapolis'

#    END AS Jurisdiction,

#    COUNT(*) AS park_count,

# SUM(PARKAREA) AS total_ParkArea ]]]------- i wrote those becuase those are what I want to display later right but it doesetn tget 
# execute now cuz theres no data the i wrote from pars cuz thats the irst step sql needs to know here the data from then where 
# to filter then groupby to short the bins base on teh case above and then the select is actually execute here where it say ok show 
# all the new column jurisdiction and also show parkcount colum and total acress colum but here the sql is smart the count and sum here
# automatically knows it is laready groubed cuz thats the order of sql. select is just written fitst to see what we want ot display but
# it igets exdcuted last and order by is the final final thing done right after select or before



parkCounts = pd.read_sql_query(query, conn)
print(parkCounts.head())

  Jurisdiction  park_count  total_ParkArea
0         None           4           521.0
1      concord           5            87.0
2   kannapolis           1            40.0


In [ ]:
#GOAL: I want to split the Parks Table into 2 tables. One for Spatial Data and the other for ADDRESS

all_data = pd.read_sql("SELECT * FROM parks", conn)

#Creating the spatial table.

columns_spatial = ['FACILITYID', 'NAME', 'X', 'Y', 'FULLADDR']
spatial_df = all_data[columns_spatial].copy()
spatial_df.to_sql('Spatial_parks', conn, if_exists = 'replace', index = False)

#Creating the attribute table
columns_atribute =['FACILITYID', 'PARKAREA', 'OPERDAYS']
atribute_df = all_data[columns_atribute].copy()

#THEN i want to make 10 percent of data pmissing to simulate real world data.

atribute_df = atribute_df.sample(frac = .9)
atribute_df.to_sql('Atribute_parks', conn, if_exists = 'replace', index = False)


#USE DISPLAY for better visuals but print works fine too. MAKE SURE 
#TO USE PRINT OR JSUT GET RID OF IT BEFORE EXPORTING AS PYTHON SCRIPT

print("SPATIAL TABLE")
print(pd.read_sql("SELECT * FROM Spatial_parks", conn).head())

print("\nATRIBUTE TABLE")
print(pd.read_sql("SELECT * FROM Atribute_parks", conn).head())

SPATIAL TABLE
  FACILITYID                                             NAME             X              Y                             FULLADDR
0      FAC-1  Bakers Creek Park/Greenway - City of Kannapolis  1.515174e+06  644107.563114         1275 West A St, 704-920-4343
1      FAC-6               Frank Liske Park - Cabarrus County  1.517086e+06  590589.513982         4001 Stough Rd, 704-920-3350
2     FAC-19                                  McAllister Park  1.572810e+06  603840.000112                         8595 Park Dr
3     FAC-12   Hartsell Park and Rec Center - City of Concord  1.525360e+06  601560.000571  60 Hartsell School Rd, 704-920-5600
4      FAC-4                   Camp Spencer - Cabarrus County  1.556115e+06  626720.964439          3155 Rimer Rd, 704-920-3350

ATRIBUTE TABLE
  FACILITYID  PARKAREA   OPERDAYS
0    FAC-164       NaN  Sun - Sat
1     FAC-22       NaN  Sun - Sat
2    FAC-179       NaN  Sun - Sat
3    FAC-177       NaN  Sun - Sat
4      FAC-4      50.0  Sun - Sa